# 利用rating2inter.ipynb中U/I的index对features进行一一对应(meta-text)
- Reindex item feature ID with IDs generated in 0rating2inter.ipynb

In [1]:
import os
import pandas as pd

In [2]:
os.chdir('/home/minhle/CaMRec/data')
os.getcwd()

'/home/minhle/CaMRec/data'

In [3]:
# load item mapping
i_id_mapping = 'i_id_mapping.csv'
df = pd.read_csv(i_id_mapping, sep='\t')
print(f'shape: {df.shape}')
df[:4]

shape: (7050, 2)


,asin,itemID
0,097293751X,0
1,9729375011,1
2,B00000IZQI,2
3,B00000J3LL,3


In [6]:

import gzip, json
meta_file = '../raw_data/meta_Baby.json.gz'

print('0 Extracting U-I interactions.')

def parse(path):
  g = gzip.open(path, 'rb')
  for l in g:
    yield eval(l)

def getDF(path):
  i = 0
  df = {}
  for d in parse(path):
    df[i] = d
    i += 1
  return pd.DataFrame.from_dict(df, orient='index')

meta_df = getDF(meta_file)

print(f'Total records: {meta_df.shape}')
meta_df[:3]

0 Extracting U-I interactions.
Total records: (71317, 9)


,asin,categories,description,title,price,imUrl,brand,related,salesRank
0,0188399313,[[Baby]],Wee-Go Glass baby bottles by LifeFactory (Baby...,Lifefactory 4oz BPA Free Glass Baby Bottles - ...,69.99,http://ecx.images-amazon.com/images/I/41Swthpd...,Lifefactory,"{'also_bought': ['B002SG7K7A', 'B003CJSXW8', '...",NaN
1,0188399518,[[Baby]],The Planet Wise Flannel Wipes are 10 super sof...,Planetwise Flannel Wipes,15.95,http://ecx.images-amazon.com/images/I/41otjnA4...,Planet Wise,"{'also_bought': ['B00G96N3YY', 'B003XSEV2O', '...",NaN
2,0188399399,[[Baby]],The Planet Wise Wipe PouchTM features our pate...,Planetwise Wipe Pouch,10.95,http://ecx.images-amazon.com/images/I/61x8h9u6...,NaN,"{'also_bought': ['B005WWI0DA', 'B005WWIMGA', '...",NaN


In [7]:
# remapping
map_dict = dict(zip(df['asin'], df['itemID']))

meta_df['itemID'] = meta_df['asin'].map(map_dict)
meta_df.dropna(subset=['itemID'], inplace=True)
meta_df['itemID'] = meta_df['itemID'].astype('int64')
#meta_df['description'] = meta_df['description'].fillna(" ")
meta_df.sort_values(by=['itemID'], inplace=True)

print(f'shape: {meta_df.shape}')
meta_df[:2]

shape: (7050, 10)


,asin,categories,description,title,price,imUrl,brand,related,salesRank,itemID
7,097293751X,[[Baby]],Easily keep track of your baby's or child's da...,"Baby Tracker&reg; - Daily Childcare Journal, S...",17.00,http://ecx.images-amazon.com/images/I/41Bb6wf%...,Time Too,"{'also_bought': ['9729375011', 'B004FN1AE8', '...",NaN,0
108,9729375011,[[Baby]],This is version of the award-winningBaby Track...,Newborn Baby Tracker&reg; - Round the Clock Ch...,15.95,http://ecx.images-amazon.com/images/I/51r3BLpL...,NaN,"{'also_bought': ['B000V5KPZ4', 'B001F8TLLU', '...",NaN,1


In [8]:
ori_cols = meta_df.columns.tolist()

ret_cols = [ori_cols[-1]] + ori_cols[:-1]
print(f'new column names: {ret_cols}')

new column names: ['itemID', 'asin', 'categories', 'description', 'title', 'price', 'imUrl', 'brand', 'related', 'salesRank']


In [9]:
meta_df[:3]

,asin,categories,description,title,price,imUrl,brand,related,salesRank,itemID
7,097293751X,[[Baby]],Easily keep track of your baby's or child's da...,"Baby Tracker&reg; - Daily Childcare Journal, S...",17.00,http://ecx.images-amazon.com/images/I/41Bb6wf%...,Time Too,"{'also_bought': ['9729375011', 'B004FN1AE8', '...",NaN,0
108,9729375011,[[Baby]],This is version of the award-winningBaby Track...,Newborn Baby Tracker&reg; - Round the Clock Ch...,15.95,http://ecx.images-amazon.com/images/I/51r3BLpL...,NaN,"{'also_bought': ['B000V5KPZ4', 'B001F8TLLU', '...",NaN,1
134,B00000IZQI,[[Baby]],This colorful car collection develops motor sk...,Fisher Price Nesting Action Vehicles,8.37,http://ecx.images-amazon.com/images/I/51E83QCC...,NaN,"{'also_bought': ['B0042D69W4', 'B00428LIZM', '...",NaN,2


In [10]:
ret_df = meta_df[ret_cols]
# dump
ret_df.to_csv(os.path.join('./', 'meta-baby.csv'), index=False)
print('done!')

done!


## Reload

In [11]:
indexed_df = pd.read_csv('meta-baby.csv')
print(f'shape: {indexed_df.shape}')
indexed_df[:4]

shape: (7050, 10)


,itemID,asin,categories,description,title,price,imUrl,brand,related,salesRank
0,0,097293751X,[['Baby']],Easily keep track of your baby's or child's da...,"Baby Tracker&reg; - Daily Childcare Journal, S...",17.00,http://ecx.images-amazon.com/images/I/41Bb6wf%...,Time Too,"{'also_bought': ['9729375011', 'B004FN1AE8', '...",NaN
1,1,9729375011,[['Baby']],This is version of the award-winningBaby Track...,Newborn Baby Tracker&reg; - Round the Clock Ch...,15.95,http://ecx.images-amazon.com/images/I/51r3BLpL...,NaN,"{'also_bought': ['B000V5KPZ4', 'B001F8TLLU', '...",NaN
2,2,B00000IZQI,[['Baby']],This colorful car collection develops motor sk...,Fisher Price Nesting Action Vehicles,8.37,http://ecx.images-amazon.com/images/I/51E83QCC...,NaN,"{'also_bought': ['B0042D69W4', 'B00428LIZM', '...",NaN
3,3,B00000J3LL,[['Baby']],This darling cloth book offers hands-on experi...,"My Quiet Book, Fabric Activity Book for Children",27.00,http://ecx.images-amazon.com/images/I/51GoNXhB...,NaN,"{'also_bought': ['B00000J3LC', 'B0043G4JOA', '...",NaN


In [12]:
## Reload

i_uni = indexed_df['itemID'].unique()

print(f'# of unique items: {len(i_uni)}')

print('min/max of unique learners: {0}/{1}'.format(min(i_uni), max(i_uni)))

# of unique items: 7050
min/max of unique learners: 0/7049
